# DocParse — Complete Guide

DocParse extracts structured fields from documents — invoices, receipts, contracts, loan applications — using LLM-powered schema extraction. This notebook covers:

- Extracting with built-in schemas
- Defining a custom schema
- Async jobs (submit → poll → result)
- Webhooks for long documents
- Local extraction with LLMExtractor
- Real-world: invoice processing pipeline

## Setup

In [ ]:
%pip install 'ag-docparse>=0.6' -q

In [ ]:
import os
AG_API_KEY = os.environ['AG_API_KEY']
AG_API_URL = os.environ['AG_API_URL']  # your ag-platform deployment

## 1. Built-in schemas

ag ships schemas for common document types. Pass `schema_name` to use one directly.

In [ ]:
from ag_docparse import AsyncJobClient, REGISTRY

print('Available schemas:')
for name, schema in REGISTRY.items():
    fields = ', '.join(f.name for f in schema.fields[:4])
    print(f'  {name:<25} fields: {fields}...')

## 2. Extract from an invoice

In [ ]:
client = AsyncJobClient(api_key=AG_API_KEY, base_url=AG_API_URL)

invoice_text = """
INVOICE
Invoice Number: INV-2024-0099
Invoice Date:   2024-11-01
Due Date:       2024-11-30

Vendor: TechSupply GmbH
        Berliner Str. 12, 10115 Berlin, Germany
        VAT ID: DE123456789

Bill To: Acme Corp
         100 Industrial Ave, London EC1A 1BB, UK

Description                  Qty   Unit Price    Line Total
Server Hardware (rack unit)    2   €4,500.00     €9,000.00
Installation & Config          1   €1,200.00     €1,200.00
Annual Maintenance             1     €600.00       €600.00

                                      Subtotal:  €10,800.00
                                    VAT (19%):   €2,052.00
                                  TOTAL DUE:  €12,852.00

Payment: Bank transfer within 30 days
IBAN: DE89 3704 0044 0532 0130 00
"""

job = client.submit(invoice_text, schema_name='invoice')
print('Job ID:', job['job_id'], '| Status:', job['status'])

In [ ]:
# Wait for completion (polls every 2 s, up to 60 s)
result = client.wait(job['job_id'], timeout=60)

print(f'Status: {result["status"]}\n')
for field, val in result['result'].items():
    conf = val.get('confidence', 0)
    print(f'  {field:<30} {str(val["value"]):<30} conf={conf:.0%}')

## 3. Custom schema

Define exactly the fields you need with `FieldSpec` and `ExtractionSchema`.

In [ ]:
from ag_docparse import FieldSpec, ExtractionSchema

purchase_order_schema = ExtractionSchema(
    name='purchase_order',
    fields=[
        FieldSpec(name='po_number',       description='Purchase order number',           type='string'),
        FieldSpec(name='order_date',       description='Date the order was placed',        type='date'),
        FieldSpec(name='supplier_name',    description='Name of the supplier',             type='string'),
        FieldSpec(name='buyer_name',       description='Name of the buying organisation',  type='string'),
        FieldSpec(name='line_items',       description='List of ordered items with qty',   type='list'),
        FieldSpec(name='total_amount',     description='Total order value including tax',  type='number'),
        FieldSpec(name='currency',         description='Currency code (ISO 4217)',         type='string'),
        FieldSpec(name='delivery_address', description='Shipping/delivery address',        type='string'),
    ]
)

po_text = """
PURCHASE ORDER  PO-2024-5512
Date: 15 October 2024
From: Omega Retail Ltd, 5 Commerce Lane, Dublin, Ireland
To:   FastShip Logistics, Unit 3, Warsaw Industrial Park, Poland

Items ordered:
  - Warehouse shelving units x 50   @ €120 each = €6,000
  - Pallet jacks x 10               @ €350 each = €3,500

Total (incl. 23% VAT):  €11,685
Deliver to: Omega Warehouse, Naas Road, Dublin 12
"""

job = client.submit(po_text, fields=[f.model_dump() for f in purchase_order_schema.fields])
result = client.wait(job['job_id'], timeout=60)

for field, val in result['result'].items():
    print(f'  {field:<20} {str(val["value"])[:50]}')

## 4. List recent jobs

In [ ]:
jobs = client.list_jobs()
print(f'{len(jobs)} recent jobs:')
for j in jobs[:5]:
    print(f'  {j["job_id"]}  status={j["status"]}  schema={j.get("schema_name", "custom")}')

## 5. Webhook for long documents

For large PDFs that take > 30 seconds, pass a `webhook_url`. ag will POST the completed result to your URL instead of blocking.

In [ ]:
# job = client.submit(
#     text=large_pdf_text,
#     schema_name='contract',
#     webhook_url='https://yourapp.com/webhooks/docparse',
# )
# print('Job queued:', job['job_id'])
# # Your webhook endpoint receives: {job_id, status, result, ...}

## 6. Load from a file

In [ ]:
from ag_docparse import load

# Works with .pdf, .txt, .png, .jpg
# doc = load('path/to/invoice.pdf')
# print(doc.text[:200])
#
# job = client.submit(doc.text, schema_name='invoice')
# result = client.wait(job['job_id'])

## 7. Built-in schema reference

| Schema name | Key fields |
|-------------|------------|
| `invoice` | invoice_number, invoice_date, vendor_name, total_gross_worth, line_items |
| `loan_application` | applicant_name, loan_amount, purpose, income, credit_score |
| `w2` | employer_name, wages, federal_tax_withheld, ssn |
| `nda` | parties, effective_date, confidentiality_period, governing_law |
| `contract` | parties, effective_date, payment_terms, termination_clause |

## 8. Real-world: invoice processing pipeline

In [ ]:
import json

def process_invoice(text: str) -> dict:
    """Submit an invoice, wait for extraction, return key fields."""
    job = client.submit(text, schema_name='invoice')
    result = client.wait(job['job_id'], timeout=90)

    if result['status'] != 'completed':
        raise RuntimeError(f'Extraction failed: {result["status"]}')

    fields = result['result']
    return {
        'invoice_number': fields.get('invoice_number', {}).get('value'),
        'vendor':         fields.get('vendor_name',    {}).get('value'),
        'total':          fields.get('total_gross_worth', {}).get('value'),
        'due_date':       fields.get('due_date',        {}).get('value'),
        'confidence':     min(v.get('confidence', 1) for v in fields.values()),
    }

summary = process_invoice(invoice_text)
print(json.dumps(summary, indent=2))